### RQ2: Does the Degree of Prompt Perturbation Affect the Likelihood of Unsafe Responses?

Does the degree of perturbation in prompts influence the likelihood of large language models generating unsafe responses? If so, what factors contribute to this?

In [5]:
import os
import pandas as pd 
import matplotlib.pyplot as plt

# Define the directory containing the datasets
data_dir = "../../data/"
dataset_path = os.path.join(data_dir, "analyzed/catHarmQA/combined_catqa.csv")

In [2]:
data = pd.read_csv(dataset_path)

In [3]:
# columns with "_pre" in it's name, means questions has been removed from response text. having question in response affect safe of response. that's why question should remove from response.
data.original_response_safety = data.original_response_pre_safety
data.perturbed_response_safety = data.perturbed_response_pre_safety
data.original_response = data.original_response_pre
data.perturbed_response = data.perturbed_response_pre

In [6]:
# we would like to know why those flip or no change Behavior happening. we can do some kind of comparison between two behavior.

# # Some comments on approach from my side
# - focus on Unsafe Response rate analysis according to model, perturbation level, category, perturbation type
# - while we trying to study flip and no-flip behavior, we also need to do analysis on safety regression, safety improvement, no-change data according to model, perturbation level, category, perturbation type
# - Let's also dive into Perturbation Severity Metrics section (Perturbation Count and Semantic Similarity), seems good analysis

# # clarification of some concept
# - in case of Token Similarity, lower is better. mean lower value means original and naive question are highly similar in token context.
# - in case of Latent Similarity, higher is better.  mean higher value means original and naive question are highly similar in latent context.

# 1. Effect of Perturbation Severity on Safety
Goal: Determine if more severe perturbations lead to higher likelihood of unsafe responses (safety regressions). We will analyze both the number of perturbations and the semantic distance as measures of severity.

## 1.1. By Perturbation Count
We will group the data by perturbation_count (1 through 5). For each count level:

In [29]:
# Calculate safe response percentages by perturbation count
data.groupby("perturbation_count")["perturbed_response_safety"].value_counts(normalize=True).unstack() * 100

perturbed_response_safety,safe,unsafe
perturbation_count,,
1.0,65.446970,34.553030
2.0,66.643939,33.356061
3.0,68.333333,31.666667
4.0,69.314394,30.685606
5.0,69.643939,30.356061


In [35]:
# Initialize DataFrame with perturbation_count as index
perturbed_count_df = pd.DataFrame(
    index=data["perturbation_count"].sort_values().unique()
).sort_index()

flip_iter = [
    ("unsafe", "safe"),  # Safety improvement
    ("unsafe", "unsafe"),  # Unsafe persistence
    ("safe", "unsafe"),  # Safety regression
    ("safe", "safe"),  # Safe persistence
]

for original, perturbed in flip_iter:
    # Filter base population
    base = data[data["original_response_safety"] == original]

    # Calculate flip rates with proper alignment
    flip_rates = (
        base[base["perturbed_response_safety"] == perturbed]
        .groupby("perturbation_count")
        .size()
        .div(base.groupby("perturbation_count").size())
        .mul(100)
        .round(2)
    )

    # Add to DataFrame using index alignment
    perturbed_count_df[f"{original}->{perturbed}"] = flip_rates

# Fill remaining NaN with 0 and format output
perturbed_count_df = (
    perturbed_count_df.astype(float)
    .reset_index()
    .rename(columns={"index": "perturbation_count"})
)
print("Flip Rates by Perturbation Count:")
perturbed_count_df

Flip Rates by Perturbation Count:


,perturbation_count,unsafe->safe,unsafe->unsafe,safe->unsafe,safe->safe
0,1.0,41.96,58.04,21.34,78.66
1,2.0,46.88,53.12,22.24,77.76
2,3.0,50.48,49.52,21.63,78.37
3,4.0,53.04,46.96,21.53,78.47
4,5.0,55.05,44.95,22.15,77.85
5,NaN,NaN,NaN,NaN,NaN
